In [1]:
# import os
# from secret_keys import OPENAI_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import langchain_core
from langchain_ollama import ChatOllama
from config_loader import load_config_as_dict

In [2]:
llm_config = load_config_as_dict()['llm']

llm_model = ChatOllama(
    model=llm_config['model'],
    temperature=llm_config['temperature'],
    num_predict=llm_config['max_tokens'],  # Ollama's equivalent of Groq's max_tokens
)
llm_model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, model='gemma2:9b', num_predict=1024, temperature=0.5)

In [3]:
query = "I want to open a restaurant for Indian Food. Suggest a fancy name for that."
name = llm_model.invoke(query)
print(name)

content="Here are some fancy names for an Indian restaurant, playing with different themes:\n\n**Elegant & Evocative:**\n\n* **Spice & Silk:**  Combines the essence of Indian cuisine with luxurious imagery.\n* **The Saffron Moon:**  Romantic and mystical, referencing a key spice and the moon's beauty.\n* **Taj Mahal Bistro:**  A classic reference with a touch of modern dining.\n* **Masala Mystique:**  Intriguing and hints at the depth of flavors.\n* **The Jewel of Agra:**  Evokes the grandeur of Mughal architecture and Indian gems.\n\n**Modern & Chic:**\n\n* **Chai & Chutney:**  Playful and contemporary, highlighting popular Indian elements.\n* **IndiGo:**  Short, catchy, and hints at both India and indigo dye.\n* **The Curry Leaf:**  Simple yet sophisticated, referencing a key ingredient.\n* **Spice Route Kitchen:**  Modern and alludes to the historical trade routes.\n* **Bombay Nights:**  Energetic and captures the vibrant spirit of Mumbai.\n\n**Unique & Playful:**\n\n* **Tandoori Ta

In [19]:
from IPython.display import display, Markdown

def llm_output_parser(output: langchain_core.messages.ai.AIMessage, text: bool = False):
    if not output:
        raise ValueError("Please share the LLM Output")
    return Markdown(output.text if text else output.content)

display(llm_output_parser(name))

Here are some fancy names for an Indian restaurant, playing with different themes:

**Elegant & Evocative:**

* **Spice & Silk:**  Combines the essence of Indian cuisine with luxurious imagery.
* **The Saffron Moon:**  Romantic and mystical, referencing a key spice and the moon's beauty.
* **Taj Mahal Bistro:**  A classic reference with a touch of modern dining.
* **Masala Mystique:**  Intriguing and hints at the depth of flavors.
* **The Jewel of Agra:**  Evokes the grandeur of Mughal architecture and Indian gems.

**Modern & Chic:**

* **Chai & Chutney:**  Playful and contemporary, highlighting popular Indian elements.
* **IndiGo:**  Short, catchy, and hints at both India and indigo dye.
* **The Curry Leaf:**  Simple yet sophisticated, referencing a key ingredient.
* **Spice Route Kitchen:**  Modern and alludes to the historical trade routes.
* **Bombay Nights:**  Energetic and captures the vibrant spirit of Mumbai.

**Unique & Playful:**

* **Tandoori Tales:**  Storytelling element, hinting at the rich history of Indian food.
* **The Masala Monk:**  Whimsical and alludes to the spiritual side of Indian culture.
* **Currylicious:**  Fun and memorable, with a focus on the deliciousness.
* **Naan Stop:**  Playful pun, highlighting a beloved Indian bread.
* **Biryani Bliss:**  Descriptive and emphasizes a popular dish.

**Tips for Choosing:**

* **Consider your target audience:** Who are you trying to attract?
* **Reflect your restaurant's concept and style:**  Is it modern, traditional, casual, or upscale?
* **Keep it memorable and easy to pronounce:**
* **Check for availability:** Make sure the name isn't already taken.





### Prompts

In [20]:
from langchain_classic.prompts import PromptTemplate

query = "I want to open a restaurant for {cuisine} food. Suggest a fancy name for that. ** SHARE JUST THE NAMES**, nothing else."

prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template=query
)

prompt_template_name.format(cuisine='Indian')

'I want to open a restaurant for Indian food. Suggest a fancy name for that. ** SHARE JUST THE NAMES**, nothing else.'

### Chains

In [24]:
from langchain_classic.chains.llm import LLMChain

llm_chain = LLMChain(llm=llm_model, prompt=prompt_template_name)
# llm_chain.run('American')
llm_chain.invoke('American')

{'cuisine': 'American',
 'text': 'The Harvest Table\nAmericana Grill\nLiberty & Oak\nThe Union Kitchen\nStateside Bistro\nCopper & Rye\nRedwood & Vine\nThe Hearth & Home\nMain Street Provisions\nEmber & Ash  \n'}

In [28]:
query = "I want to open a restaurant for {cuisine} food. Suggest a fancy name for that. ** SHARE JUST THE BEST NAME** -- only 1, nothing else."
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template=query
)
rest_name_chain = LLMChain(llm=llm_model, prompt=prompt_template_name)

prompt_template_items = PromptTemplate(
    input_variables=['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}. Return it as a comma-separated list."
)
food_items_chain = LLMChain(llm=llm_model, prompt=prompt_template_items)

In [29]:
# for 1 input -> 1 ouput => use SimpleSequentialChain
from langchain_classic.chains.sequential import SimpleSequentialChain

chain = SimpleSequentialChain(chains=[rest_name_chain, food_items_chain])
chain.invoke("India")

{'input': 'India',
 'output': 'Tandoori Chicken, Butter Chicken, Rogan Josh, Vindaloo, Saag Paneer, Samosas, Naan, Biryani, Chana Masala, Dal Makhani, Tikka Masala, Mango Lassi, Gulab Jamun  \n\n\n'}

In [34]:
query = "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this. ** SHARE JUST THE BEST NAME** -- only 1, nothing else."
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template=query
)
rest_name_chain = LLMChain(llm=llm_model, prompt=prompt_template_name, output_key='restaurant_name')

prompt_template_items = PromptTemplate(
    input_variables=['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}. Return it as a comma-separated list. Share ** just the items** nothing else."
)
food_items_chain = LLMChain(llm=llm_model, prompt=prompt_template_items, output_key='menu_items')

In [35]:
# for more than 1 input -> more than 1 output => use SequentialChain
from langchain_classic.chains.sequential import SequentialChain

chain = SequentialChain(
    chains=[rest_name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaurant_name', 'menu_items']
)
chain.invoke({'cuisine': 'Arabic'})

{'cuisine': 'Arabic',
 'restaurant_name': 'Al-Layali Al-Mahboba  \n',
 'menu_items': 'Hummus, Baba Ghanoush, Falafel, Shawarma, Chicken Kabob, Lamb Kofta,  Grilled Salmon, Maqluba, Baklava, Knafeh \n\n\n'}

___
___